# HW03 — Model Serving & Deployment

Your model is trained and tracked in MLflow. A model that only runs in a notebook is not useful in production.

In this homework you will:

- verify that your trained model is reproducible and ready for serving
- wrap it in a FastAPI service with proper input validation and batch support
- measure the performance difference between single and batch prediction
- package the service in Docker with a size-optimized image
- write Kubernetes manifests to deploy the service

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords, tokens, or notebook checkpoints.
Do not hardcode passwords anywhere in your code.

## Useful references

- MLflow model loading: https://mlflow.org/docs/latest/python_api/mlflow.sklearn.html
- FastAPI: https://fastapi.tiangolo.com/
- FastAPI lifespan: https://fastapi.tiangolo.com/advanced/events/
- Pydantic v2: https://docs.pydantic.dev/latest/
- Dockerfile reference: https://docs.docker.com/reference/dockerfile/
- Docker multi-stage builds: https://docs.docker.com/build/building/multi-stage/
- Kubernetes Deployments: https://kubernetes.io/docs/concepts/workloads/controllers/deployment/
- Kubernetes Services: https://kubernetes.io/docs/concepts/services-networking/service/

## What to avoid

- Loading the model inside the request handler. Load once at startup.
- Hardcoded passwords in any source file.
- A Docker image that bakes in the model file. Pull from MLflow at startup.
- Returning raw numpy types from the API. JSON needs native Python types.
- Skipping the batch vs single benchmark. The numbers tell a story.

In [1]:
import os
import re
import time
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

PROJECT = Path.cwd()
for path in ['src/airbnb_serving', 'k8s', 'reports', 'screenshots']:
    (PROJECT / path).mkdir(parents=True, exist_ok=True)
(PROJECT / 'src/airbnb_serving/__init__.py').write_text('__version__ = "0.1.0"\n')

STUDENT_ID = os.getenv('QBC12_STUDENT_ID', '') or input('GitHub username / student id: ').strip()
safe_student = re.sub(r'[^a-zA-Z0-9_]', '_', STUDENT_ID.lower())
EXPERIMENT_NAME = f'qbc12_hw02_{safe_student}'

print('PROJECT:', PROJECT)
print('EXPERIMENT_NAME:', EXPERIMENT_NAME)

PROJECT: /Users/parhamrahimi/Documents/bluBank/Quera/HW3
EXPERIMENT_NAME: qbc12_hw02_parhamrahimi


---
## Part 1 — Model Reproducibility Check

Before serving a model, you must verify it produces exactly the same output as it did during training.

This is called a **reproducibility check** and it catches silent bugs like:
- preprocessing mismatch between training and serving
- wrong model version loaded
- feature column order changed

### 1.1 Connect to MLflow and load your best model

In [2]:
MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI', 'http://185.50.38.163:33014')
MLFLOW_USERNAME = os.getenv('MLFLOW_TRACKING_USERNAME', '') or input('MLflow username: ').strip()
MLFLOW_PASSWORD = os.getenv('MLFLOW_TRACKING_PASSWORD', '') or input('MLflow password: ').strip()

os.environ['MLFLOW_TRACKING_USERNAME'] = MLFLOW_USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    raise ValueError(f'Experiment not found: {EXPERIMENT_NAME}. Complete HW02 first.')

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.leakage_status = 'clean' and tags.selected_for_serving = 'true'",
    order_by=['metrics.f1 DESC'],
)

if runs.empty:
    raise ValueError(
        'No run tagged selected_for_serving=true found. '
        'Go to MLflow UI, find your best clean run, and add the tag.'
    )

BEST_RUN_ID = runs.iloc[0]['run_id']
MODEL_URI = f'runs:/{BEST_RUN_ID}/model'

print('Best run ID  :', BEST_RUN_ID)
print('Model URI    :', MODEL_URI)
print('Run name     :', runs.iloc[0].get('tags.mlflow.runName'))
print('F1 score     :', runs.iloc[0].get('metrics.f1'))

ValueError: Experiment not found: qbc12_hw02_parhamrahimi. Complete HW02 first.

In [3]:
model = mlflow.sklearn.load_model(MODEL_URI)
print('Model type:', type(model))
print('Model steps:', list(model.named_steps.keys()) if hasattr(model, 'named_steps') else 'not a pipeline')

NameError: name 'MODEL_URI' is not defined

### 1.2 Load your HW01 feature dataset

You will use a small sample from your HW01 feature parquet file to verify reproducibility.

In [4]:
FEATURE_COLS = [
    'room_type', 'property_type', 'neighbourhood_name',
    'accommodates', 'bedrooms', 'beds', 'bathrooms', 'listing_price',
    'minimum_nights', 'maximum_nights', 'instant_bookable', 'host_is_superhost',
    'host_listing_count', 'total_reviews_before_cutoff', 'unique_reviewers_before_cutoff',
    'avg_comment_len_before_cutoff', 'max_comment_len_before_cutoff',
    'days_since_last_review', 'available_days_last_90d', 'available_rate_last_90d',
    'avg_minimum_nights_calendar_last_90d', 'avg_maximum_nights_calendar_last_90d',
    'available_days_last_30d', 'available_rate_last_30d',
    'avg_minimum_nights_calendar_last_30d', 'avg_maximum_nights_calendar_last_30d',
]
TARGET_COL = 'high_demand_proxy'

# Load your HW01 parquet file
# Adjust the path if needed
parquet_path = list(Path('data/features').glob('*.parquet'))
if not parquet_path:
    raise FileNotFoundError('HW01 feature parquet not found. Run HW01 ETL first.')

df = pd.read_parquet(parquet_path[0])
print('Dataset shape:', df.shape)
df[FEATURE_COLS + [TARGET_COL]].head(3)

Dataset shape: (10480, 33)


KeyError: "['host_is_superhost'] not in index"

### 1.3 Reproducibility check

**TODO 1.3**

Take a sample of 50 rows from the dataset.

Run `model.predict()` on those rows **twice** and verify the results are identical.

Then compare the predictions against the `high_demand_proxy` column and print:
- how many predictions match the training label
- the accuracy on this sample

If both runs produce identical output, print `Reproducibility check passed.`
If they differ, raise a `ValueError`.

In [5]:
# TODO 1.3 — Reproducibility check
sample = df[FEATURE_COLS + [TARGET_COL]].sample(50, random_state=42)
X_sample = sample[FEATURE_COLS]
y_sample = sample[TARGET_COL]

# Run predict twice and verify identical results
preds1 = model.predict(X_sample)
preds2 = model.predict(X_sample)

if not np.array_equal(preds1, preds2):
    raise ValueError('Reproducibility check failed: two runs produced different predictions.')

matches = (preds1 == y_sample).sum()
accuracy = matches / len(y_sample)
print(f'Predictions matching the training label: {matches}/{len(y_sample)}')
print(f'Accuracy on this sample: {accuracy:.4f}')
print('Reproducibility check passed.')

KeyError: "['host_is_superhost'] not in index"

---
## Part 2 — FastAPI Service

A REST API is the standard way to expose an ML model to other systems.

You will build a FastAPI app with two prediction endpoints:
- `POST /predict` — single listing prediction
- `POST /predict/batch` — multiple listings in one request

Then you will measure how much faster batch is compared to calling single predict in a loop.

### 2.1 Input and output schemas

In [6]:
# TODO 2.1 — Input and output schemas
(PROJECT / 'src/airbnb_serving/schema.py').write_text(textwrap.dedent('''\
from typing import Optional

from pydantic import BaseModel


class ListingFeatures(BaseModel):
    room_type: str
    property_type: str
    neighbourhood_name: str
    accommodates: int
    bedrooms: Optional[float] = None
    beds: Optional[float] = None
    bathrooms: Optional[float] = None
    listing_price: float
    minimum_nights: int
    maximum_nights: int
    instant_bookable: bool
    host_is_superhost: bool
    host_listing_count: int
    total_reviews_before_cutoff: Optional[float] = None
    unique_reviewers_before_cutoff: Optional[float] = None
    avg_comment_len_before_cutoff: Optional[float] = None
    max_comment_len_before_cutoff: Optional[float] = None
    days_since_last_review: Optional[float] = None
    available_days_last_90d: int
    available_rate_last_90d: float
    avg_minimum_nights_calendar_last_90d: Optional[float] = None
    avg_maximum_nights_calendar_last_90d: Optional[float] = None
    available_days_last_30d: int
    available_rate_last_30d: float
    avg_minimum_nights_calendar_last_30d: Optional[float] = None
    avg_maximum_nights_calendar_last_30d: Optional[float] = None


class PredictionResponse(BaseModel):
    listing_id: Optional[int] = None
    prediction: int
    probability_high_demand: float
    model_run_id: str
''').lstrip() + '\n')
print('schema.py written.')

schema.py written.


### 2.2 Prediction logic

In [7]:
# TODO 2.2 — Prediction logic
(PROJECT / 'src/airbnb_serving/predictor.py').write_text(textwrap.dedent('''\
import pandas as pd

from airbnb_serving.schema import ListingFeatures, PredictionResponse

FEATURE_COLS = [
    "room_type", "property_type", "neighbourhood_name",
    "accommodates", "bedrooms", "beds", "bathrooms", "listing_price",
    "minimum_nights", "maximum_nights", "instant_bookable", "host_is_superhost",
    "host_listing_count", "total_reviews_before_cutoff", "unique_reviewers_before_cutoff",
    "avg_comment_len_before_cutoff", "max_comment_len_before_cutoff",
    "days_since_last_review", "available_days_last_90d", "available_rate_last_90d",
    "avg_minimum_nights_calendar_last_90d", "avg_maximum_nights_calendar_last_90d",
    "available_days_last_30d", "available_rate_last_30d",
    "avg_minimum_nights_calendar_last_30d", "avg_maximum_nights_calendar_last_30d",
]


def predict_single(features: ListingFeatures, model, run_id: str) -> PredictionResponse:
    row = features.model_dump()
    df = pd.DataFrame([row])[FEATURE_COLS]
    pred_label = int(model.predict(df)[0])
    proba = float(model.predict_proba(df)[:, 1][0])
    return PredictionResponse(
        prediction=pred_label,
        probability_high_demand=proba,
        model_run_id=run_id,
    )


def predict_batch(features_list, model, run_id):
    rows = [f.model_dump() for f in features_list]
    df = pd.DataFrame(rows)[FEATURE_COLS]
    pred_labels = model.predict(df)
    probas = model.predict_proba(df)[:, 1]
    results = []
    for pred, proba in zip(pred_labels, probas):
        results.append(PredictionResponse(
            prediction=int(pred),
            probability_high_demand=float(proba),
            model_run_id=run_id,
        ))
    return results
''').lstrip() + '\n')
print('predictor.py written.')

predictor.py written.


### 2.3 FastAPI app

In [8]:
# TODO 2.3 — FastAPI app
(PROJECT / 'src/airbnb_serving/app.py').write_text(textwrap.dedent('''\
import os
from contextlib import asynccontextmanager
from typing import List

import mlflow
from fastapi import FastAPI

from airbnb_serving.schema import ListingFeatures, PredictionResponse
from airbnb_serving.predictor import predict_single, predict_batch


MODEL_RUN_ID: str = ""
model = None


@asynccontextmanager
async def lifespan(app: FastAPI):
    global model, MODEL_RUN_ID
    MODEL_RUN_ID = os.environ["MODEL_RUN_ID"]
    tracking_uri = os.environ["MLFLOW_TRACKING_URI"]
    username = os.environ["MLFLOW_TRACKING_USERNAME"]
    password = os.environ["MLFLOW_TRACKING_PASSWORD"]
    os.environ["MLFLOW_TRACKING_USERNAME"] = username
    os.environ["MLFLOW_TRACKING_PASSWORD"] = password
    mlflow.set_tracking_uri(tracking_uri)
    model = mlflow.sklearn.load_model(f"runs:/{MODEL_RUN_ID}/model")
    print(f"Model loaded from MLflow run {MODEL_RUN_ID}")
    yield


app = FastAPI(
    title="Airbnb Serving API",
    description="Serves the HW02 model for predicting high-demand listings.",
    version="1.0.0",
    lifespan=lifespan,
)


@app.get("/health")
def health():
    return {"status": "ok", "model_run_id": MODEL_RUN_ID}


@app.post("/predict", response_model=PredictionResponse)
def predict(features: ListingFeatures):
    return predict_single(features, model, MODEL_RUN_ID)


@app.post("/predict/batch", response_model=List[PredictionResponse])
def batch_predict(features_list: List[ListingFeatures]):
    return predict_batch(features_list, model, MODEL_RUN_ID)
''').lstrip() + '\n')
print('app.py written.')

app.py written.


### 2.4 Package metadata

In [9]:
# TODO 2.4 — Package metadata
(PROJECT / 'pyproject.toml').write_text(textwrap.dedent('''\
[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "airbnb-serving"
version = "0.1.0"
description = "Airbnb high-demand prediction serving API"
requires-python = ">=3.10"
dependencies = [
    "fastapi>=0.111",
    "uvicorn[standard]>=0.29",
    "mlflow>=2.13",
    "scikit-learn>=1.4",
    "pandas>=2.0",
    "pydantic>=2.0",
    "python-dotenv>=1.0",
]

[tool.setuptools.packages.find]
where = ["src"]
''').lstrip() + '\n')

(PROJECT / 'requirements.txt').write_text(textwrap.dedent('''\
fastapi>=0.111
uvicorn[standard]>=0.29
mlflow>=2.13
scikit-learn>=1.4
pandas>=2.0
pydantic>=2.0
python-dotenv>=1.0
''').lstrip() + '\n')

print('pyproject.toml and requirements.txt written.')

pyproject.toml and requirements.txt written.


### 2.5 Local install and smoke test

Install the package and manually start the server in a terminal before running the test cell below.

```bash
pip install -e .

MODEL_RUN_ID=<your_run_id> \
MLFLOW_TRACKING_URI=http://185.50.38.163:33014 \
MLFLOW_TRACKING_USERNAME=<user> \
MLFLOW_TRACKING_PASSWORD=<pass> \
uvicorn airbnb_serving.app:app --host 0.0.0.0 --port 8000
```

In [10]:
import sys
!{sys.executable} -m pip install -q -e .


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


In [11]:
import subprocess, time, signal, os

server_proc = subprocess.Popen(
    ['uvicorn', 'airbnb_serving.app:app', '--host', '0.0.0.0', '--port', '12345'],
    env={**os.environ, 'MODEL_RUN_ID': BEST_RUN_ID},
)
time.sleep(15)  # wait for model to load from MLflow
print('Server started, PID:', server_proc.pid)

NameError: name 'BEST_RUN_ID' is not defined

In [12]:
import requests

BASE_URL = 'http://localhost:12345'

health = requests.get(f'{BASE_URL}/health')
assert health.status_code == 200, f'Health check failed: {health.text}'
print('Health:', health.json())

sample_payload = {
    'room_type': 'Entire home/apt',
    'property_type': 'Entire rental unit',
    'neighbourhood_name': 'Centrum-West',
    'accommodates': 2,
    'bedrooms': 1.0,
    'beds': 1.0,
    'bathrooms': 1.0,
    'listing_price': 150.0,
    'minimum_nights': 2,
    'maximum_nights': 365,
    'instant_bookable': True,
    'host_is_superhost': False,
    'host_listing_count': 1,
    'total_reviews_before_cutoff': 10.0,
    'unique_reviewers_before_cutoff': 9.0,
    'avg_comment_len_before_cutoff': 120.0,
    'max_comment_len_before_cutoff': 300.0,
    'days_since_last_review': 30.0,
    'available_days_last_90d': 45,
    'available_rate_last_90d': 0.5,
    'avg_minimum_nights_calendar_last_90d': 2.0,
    'avg_maximum_nights_calendar_last_90d': 365.0,
    'available_days_last_30d': 15,
    'available_rate_last_30d': 0.5,
    'avg_minimum_nights_calendar_last_30d': 2.0,
    'avg_maximum_nights_calendar_last_30d': 365.0,
}

resp = requests.post(f'{BASE_URL}/predict', json=sample_payload)
assert resp.status_code == 200, f'Single predict failed: {resp.text}'
print('Single predict:', resp.json())

print('Local smoke test passed.')

ConnectionError: HTTPConnectionPool(host='localhost', port=12345): Max retries exceeded with url: /health (Caused by NewConnectionError("HTTPConnection(host='localhost', port=12345): Failed to establish a new connection: [Errno 61] Connection refused"))

### 2.6 Batch vs single benchmark

**TODO 2.6**

Take 100 rows from your feature dataset.

Measure:
1. Time to call `POST /predict` 100 times in a loop (single)
2. Time to call `POST /predict/batch` once with all 100 rows (batch)

Print a comparison table with total time and time per prediction for each approach.

Then answer: why is batch faster? Write your answer as a comment in the cell.

In [13]:
# TODO 2.6 — Batch vs single benchmark
import math

def clean_for_json(row: dict) -> dict:
    return {
        k: None if isinstance(v, float) and math.isnan(v) else v
        for k, v in row.items()
    }

BENCHMARK_SIZE = 100
benchmark_rows = [
    clean_for_json(row)
    for row in df[FEATURE_COLS].head(BENCHMARK_SIZE).to_dict(orient='records')
]

# --- Single prediction loop ---
t0 = time.perf_counter()
for row in benchmark_rows:
    requests.post(f'{BASE_URL}/predict', json=row)
t1 = time.perf_counter()
single_total = t1 - t0
single_per = (single_total / BENCHMARK_SIZE) * 1000  # ms

# --- Batch prediction ---
t2 = time.perf_counter()
requests.post(f'{BASE_URL}/predict/batch', json=benchmark_rows)
t3 = time.perf_counter()
batch_total = t3 - t2
batch_per = (batch_total / BENCHMARK_SIZE) * 1000  # ms

# --- Print results ---
speedup = single_total / batch_total if batch_total > 0 else float('inf')
print(f"{'Method':<14} | {'Total (s)':<10} | {'Per prediction (ms)':<20}")
print(f"{'single loop':<14} | {single_total:<10.3f} | {single_per:<20.1f}")
print(f"{'batch':<14} | {batch_total:<10.3f} | {batch_per:<20.1f}")
print(f"Speedup: {speedup:.1f}x")

# --- Answer ---
# Why is batch faster?
# Batch prediction is faster because it sends all 100 listings in a single HTTP request,
# eliminating the per-request overhead of network round-trips, JSON serialization/deserialization,
# and HTTP connection setup. Additionally, the model processes all rows in one vectorized call
# to predict/predict_proba, which is more efficient than making 100 separate calls.

KeyError: "['host_is_superhost'] not in index"

In [14]:
server_proc.terminate()
print('Server stopped.')

NameError: name 'server_proc' is not defined

---
## Part 3 — Docker

You will build two Docker images and compare their sizes.

This teaches you that image size is not free — it affects pull time, storage cost, and attack surface.

### 3.1 Naive Dockerfile

In [15]:
# TODO 3.1
# Create Dockerfile.naive
#
# A simple, unoptimized Dockerfile:
#   FROM python:3.11
#   WORKDIR /app
#   COPY . .
#   RUN pip install -r requirements.txt && pip install -e .
#   EXPOSE 8000
#   CMD ["uvicorn", "airbnb_serving.app:app", "--host", "0.0.0.0", "--port", "8000"]

# Write your code here.

### 3.2 Optimized Dockerfile

A multi-stage build separates the build environment from the runtime environment.

Stage 1 (builder): install everything, build the package.
Stage 2 (runtime): copy only what is needed to run, nothing else.

In [16]:
# TODO 3.2
# Create Dockerfile (optimized, multi-stage)
#
# Stage 1 - builder:
#   FROM python:3.11-slim AS builder
#   install build tools, install deps into /install
#
# Stage 2 - runtime:
#   FROM python:3.11-slim
#   copy only /install from builder
#   copy src/
#   EXPOSE 8000
#   CMD uvicorn ...
#
# Also create .dockerignore to exclude:
#   .git, .venv, __pycache__, .ipynb_checkpoints, *.pyc, data/, .env

# Write your code here.

### 3.3 Build and compare image sizes

In [17]:
!docker build -f Dockerfile.naive -t qbc12-airbnb-serving:naive .
!docker build -f Dockerfile -t qbc12-airbnb-serving:optimized .

[+] Building 0.0s (0/1)                                    docker:desktop-linux


[+] Building 0.2s (7/8)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile.naive                 0.0s
 => => transferring dockerfile: 232B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.11             0.0s
 => [internal] load .dockerignore                                          0.0s
 => => transferring context: 98B                                           0.0s
 => [1/4] FROM docker.io/library/python:3.11@sha256:9139edd4aa848a1aa1c63  0.0s
 => => resolve docker.io/library/python:3.11@sha256:9139edd4aa848a1aa1c63  0.0s
 => [internal] load build context                                          0.0s
 => => transferring context: 4.69MB                                        0.0s
 => CACHED [2/4] WORKDIR /app                                              0.0s
 => [3/4] COPY . .                                                         0.0s
 => [4/4] RUN pip install -r requirement

[+] Building 0.4s (7/8)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile.naive                 0.0s
 => => transferring dockerfile: 232B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.11             0.0s
 => [internal] load .dockerignore                                          0.0s
 => => transferring context: 98B                                           0.0s
 => [1/4] FROM docker.io/library/python:3.11@sha256:9139edd4aa848a1aa1c63  0.0s
 => => resolve docker.io/library/python:3.11@sha256:9139edd4aa848a1aa1c63  0.0s
 => [internal] load build context                                          0.0s
 => => transferring context: 4.69MB                                        0.0s
 => CACHED [2/4] WORKDIR /app                                              0.0s
 => [3/4] COPY . .                                                         0.0s
 => [4/4] RUN pip install -r requirement

[+] Building 0.5s (7/8)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile.naive                 0.0s
 => => transferring dockerfile: 232B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.11             0.0s
 => [internal] load .dockerignore                                          0.0s
 => => transferring context: 98B                                           0.0s
 => [1/4] FROM docker.io/library/python:3.11@sha256:9139edd4aa848a1aa1c63  0.0s
 => => resolve docker.io/library/python:3.11@sha256:9139edd4aa848a1aa1c63  0.0s
 => [internal] load build context                                          0.0s
 => => transferring context: 4.69MB                                        0.0s
 => CACHED [2/4] WORKDIR /app                                              0.0s
 => [3/4] COPY . .                                                         0.0s
 => [4/4] RUN pip install -r requirement

[+] Building 0.7s (7/8)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile.naive                 0.0s
 => => transferring dockerfile: 232B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.11             0.0s
 => [internal] load .dockerignore                                          0.0s
 => => transferring context: 98B                                           0.0s
 => [1/4] FROM docker.io/library/python:3.11@sha256:9139edd4aa848a1aa1c63  0.0s
 => => resolve docker.io/library/python:3.11@sha256:9139edd4aa848a1aa1c63  0.0s
 => [internal] load build context                                          0.0s
 => => transferring context: 4.69MB                                        0.0s
 => CACHED [2/4] WORKDIR /app                                              0.0s
 => [3/4] COPY . .                                                         0.0s
 => [4/4] RUN pip install -r requirement

...

[+] Building 92.5s (13/14)                                 docker:desktop-linux
 => [internal] load .dockerignore                                          0.0s
 => => transferring context: 98B                                           0.0s
 => [builder 1/4] FROM docker.io/library/python:3.11-slim@sha256:c20888b6  0.0s
 => => resolve docker.io/library/python:3.11-slim@sha256:c20888b6acdd1e63  0.0s
 => [internal] load build context                                          0.0s


 => => transferring context: 1.02kB                                        0.0s
 => CACHED [builder 2/4] WORKDIR /app                                      0.0s
 => [builder 3/4] COPY requirements.txt pyproject.toml ./                  0.3s
 => [builder 4/4] RUN pip install --no-cache-dir -r requirements.txt      75.4s
 => [stage-1 3/7] COPY --from=builder /usr/local/lib/python3.11/site-pack  1.1s
 => [stage-1 4/7] COPY --from=builder /usr/local/bin /usr/local/bin/       0.1s
 => [stage-1 5/7] COPY src/ src/                                           0.0s
 => [stage-1 6/7] COPY artifacts/ artifacts/                               0.0s
 => [stage-1 7/7] COPY pyproject.toml ./                                   0.0s
 => exporting to image                                                    15.0s
 => => exporting layers                                                   12.6s
 => => exporting manifest sha256:87895594b41dee041f62f90d9114713223c4a865  0.0s
 => => exporting config sha256:d6149c77b

[+] Building 92.6s (13/14)                                 docker:desktop-linux
 => [internal] load .dockerignore                                          0.0s
 => => transferring context: 98B                                           0.0s
 => [builder 1/4] FROM docker.io/library/python:3.11-slim@sha256:c20888b6  0.0s
 => => resolve docker.io/library/python:3.11-slim@sha256:c20888b6acdd1e63  0.0s
 => [internal] load build context                                          0.0s
 => => transferring context: 1.02kB                                        0.0s
 => CACHED [builder 2/4] WORKDIR /app                                      0.0s
 => [builder 3/4] COPY requirements.txt pyproject.toml ./                  0.3s
 => [builder 4/4] RUN pip install --no-cache-dir -r requirements.txt      75.4s
 => [stage-1 3/7] COPY --f

rom=builder /usr/local/lib/python3.11/site-pack  1.1s
 => [stage-1 4/7] COPY --from=builder /usr/local/bin /usr/local/bin/       0.1s
 => [stage-1 5/7] COPY src/ src/                                           0.0s
 => [stage-1 6/7] COPY artifacts/ artifacts/                               0.0s
 => [stage-1 7/7] COPY pyproject.toml ./                                   0.0s
 => exporting to image                                                    15.1s
 => => exporting layers                                                   12.6s
 => => exporting manifest sha256:87895594b41dee041f62f90d9114713223c4a865  0.0s
 => => exporting config sha256:d6149c77bf301cc738b989a1c2b0b912f02ff3ab71  0.0s
 => => exporting attestation manifest sha256:3ee336d922c693158d29b65edebf  0.0s
 => => exporting manifest list sha256:48ce8079b656bfff54630eafe2cad2dfda3  0.0s
 => => naming to docker.io/library/qbc12-airbnb-serving:optimized          0.0s
 => => unpacking to docker.io/library/qbc12-airbnb-serving:optimiz

[+] Building 92.7s (14/14) FINISHED                        docker:desktop-linux
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 590B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.11-slim        0.0s
 => [internal] load .dockerignore                                          0.0s
 => => transferring context: 98B                                           0.0s
 => [builder 1/4] FROM docker.io/library/python:3.11-slim@sha256:c20888b6  0.0s
 => => resolve docker.io/library/python:3.11-slim@sha256:c20888b6acdd1e63  0.0s
 => [internal] load build context                                          0.0s
 => => transferring context: 1.02kB                                        0.0s
 => CACHED [builder 2/4] WORKDIR /app                                      0.0s
 => [builder 3/4] COPY requirements.txt pyproject.toml ./                  0.3s
 => [builder 4/4] RUN pip install --no-c

In [18]:
# TODO 3.3
# Run this cell after building both images.
# It compares image sizes and saves the result to reports/.

import subprocess, json

result = subprocess.run(
    ['docker', 'images', '--format', '{{json .}}'],
    capture_output=True, text=True
)

images = [json.loads(line) for line in result.stdout.strip().split('\n') if line]
serving_images = [
    img for img in images
    if img.get('Repository') == 'qbc12-airbnb-serving'
]

size_df = pd.DataFrame(serving_images)[['Repository', 'Tag', 'Size']]
print(size_df.to_string(index=False))

report_lines = [
    '# HW03 Docker Image Size Report', '',
    size_df.to_markdown(index=False), '',
    '## Analysis',
    '<!-- TODO: Write 2-3 sentences explaining why the sizes differ and which you would use in production. -->',
]
Path('reports/docker_size_report.md').write_text('\n'.join(report_lines) + '\n')
print('\nReport saved to reports/docker_size_report.md')

          Repository       Tag   Size
qbc12-airbnb-serving optimized 1.33GB
qbc12-airbnb-serving     naive 3.14GB


ImportError: Missing optional dependency 'tabulate'.  Use pip or conda to install tabulate.

### 3.4 Docker Compose

In [19]:
# TODO 3.4
# Create docker-compose.yml
#
# service name: airbnb-serving
# image: qbc12-airbnb-serving:optimized
# ports: 8000:8000
# env_file: .env
#
# Also create .env.example (no real values) to commit to Git:
#   MLFLOW_TRACKING_URI=
#   MLFLOW_TRACKING_USERNAME=
#   MLFLOW_TRACKING_PASSWORD=
#   MODEL_RUN_ID=
#
# Add .env to .gitignore

# Write your code here.

In [20]:
# Docker Compose smoke test
!docker compose up -d

import time, requests
time.sleep(8)  # wait for model to load from MLflow

health = requests.get('http://localhost:8000/health')
assert health.status_code == 200, f'Failed: {health.text}'
print('Docker Compose health check passed:', health.json())

!docker compose down

[+] up 0/1
 ⠋ Container hw3-airbnb-serving-1 Recreate                                  0.1s


[+] up 0/1
 ⠙ Container hw3-airbnb-serving-1 Recreate                                  0.2s


[+] up 0/1
 ⠹ Container hw3-airbnb-serving-1 Recreate                                  0.3s


[+] up 0/1
 ⠸ Container hw3-airbnb-serving-1 Recreate                                  0.4s


[+] up 0/1
 ⠼ Container hw3-airbnb-serving-1 Recreate                                  0.5s


...

[+] down 0/1
 ⠹ Container hw3-airbnb-serving-1 Stopping                                  0.3s


[+] down 0/1
 ⠸ Container hw3-airbnb-serving-1 Stopping                                  0.4s


[+] down 1/2
 ✔ Container hw3-airbnb-serving-1 Removed                                   0.5s
 ⠋ Network hw3_default            Removing                                  0.0s


[+] down 1/2
 ✔ Container hw3-airbnb-serving-1 Removed                                   0.5s
 ⠙ Network hw3_default            Removing                                  0.1s


[+] down 2/2
 ✔ Container hw3-airbnb-serving-1 Removed                                   0.5s
 ✔ Network hw3_default            Removed                                   0.2s


---
## Part 4 — Kubernetes Manifests

Kubernetes is the standard way to run containers in production at scale.

You do not need a real cluster for this homework. The deliverable is correct YAML files that a cluster could apply.

Key concepts you will use:

| Concept | What it does |
|---|---|
| **Pod** | Runs your container |
| **Deployment** | Manages multiple identical Pods, handles restarts |
| **Service** | Stable network endpoint that routes traffic to Pods |
| **Secret** | Stores sensitive values like passwords, not plaintext in YAML |
| **readinessProbe** | Tells Kubernetes when a Pod is ready to receive traffic |
| **resource limits** | Prevents one Pod from consuming all server memory |

### 4.1 Deployment

In [21]:
# TODO 4.1
# Create k8s/deployment.yaml
#
# Requirements:
#   apiVersion: apps/v1
#   kind: Deployment
#   name: airbnb-serving
#   replicas: 2
#   image: qbc12-airbnb-serving:optimized
#   containerPort: 8000
#
#   env vars from a Secret named airbnb-serving-secret:
#     MLFLOW_TRACKING_URI
#     MLFLOW_TRACKING_USERNAME
#     MLFLOW_TRACKING_PASSWORD
#     MODEL_RUN_ID
#
#   resources:
#     limits:   cpu: 500m, memory: 512Mi
#     requests: cpu: 100m, memory: 256Mi
#
#   readinessProbe:
#     httpGet path: /health
#     initialDelaySeconds: 15  (model needs time to load from MLflow)
#     periodSeconds: 10

# Write your code here.

### 4.2 Service

In [22]:
# TODO 4.2
# Create k8s/service.yaml
#
# Requirements:
#   apiVersion: v1
#   kind: Service
#   name: airbnb-serving
#   type: ClusterIP
#   port: 80 -> targetPort: 8000
#   selector must match the labels in your Deployment

# Write your code here.

### 4.3 Conceptual questions

Answer these in the markdown cell below. One or two sentences each is enough.

**TODO 4.3 — Answer here:**

**Q1.** We set `replicas: 2` instead of 1. What happens to traffic if one Pod crashes while replicas is 1 vs 2?

A: With 1 replica, all traffic is lost until Kubernetes restarts the Pod, causing downtime. With 2 replicas, the surviving Pod continues serving traffic while Kubernetes replaces the crashed one, providing high availability with no interruption.

**Q2.** The `readinessProbe` has `initialDelaySeconds: 15`. Why do we need a delay specifically for this service?

A: The model is loaded from MLflow at startup, which requires network calls and model deserialization that can take several seconds. Without the delay, Kubernetes would start sending traffic before the model is fully loaded, causing failed requests. The 15-second delay gives the service enough time to pull and initialize the model.

**Q3.** Why do we store credentials in a Kubernetes Secret instead of writing them directly in `deployment.yaml`?

A: Kubernetes Secrets are designed for sensitive data — they are base64-encoded, can be encrypted at rest, support RBAC access control, and are not exposed in plain text in version-controlled YAML files. Writing credentials directly in deployment.yaml would leak them to anyone with repository access.

**Q4.** What is the difference between `ClusterIP` and `LoadBalancer` service types? When would you use each?

A: ClusterIP exposes the service only within the cluster (internal communication between microservices), while LoadBalancer provisions an external load balancer (typically from a cloud provider) that exposes the service to the internet. Use ClusterIP for internal services like databases or internal APIs; use LoadBalancer when the service needs to receive traffic from outside the cluster.

---
## Final Proof

If this cell fails, HW03 is not complete.

In [23]:
required_files = [
    'src/airbnb_serving/__init__.py',
    'src/airbnb_serving/schema.py',
    'src/airbnb_serving/predictor.py',
    'src/airbnb_serving/app.py',
    'pyproject.toml',
    'requirements.txt',
    'Dockerfile',
    'Dockerfile.naive',
    'docker-compose.yml',
    '.env.example',
    '.dockerignore',
    'k8s/deployment.yaml',
    'k8s/service.yaml',
    'reports/docker_size_report.md',
]

missing = [f for f in required_files if not (PROJECT / f).exists()]
assert not missing, f'Missing files:\n' + '\n'.join(missing)

# Check .env is gitignored
gitignore = (PROJECT / '.gitignore').read_text() if (PROJECT / '.gitignore').exists() else ''
assert '.env' in gitignore, '.env must be in .gitignore'

# Check Dockerfile does not copy .env
for df_name in ['Dockerfile', 'Dockerfile.naive']:
    content = (PROJECT / df_name).read_text()
    assert 'COPY .env' not in content, f'Do not copy .env in {df_name}'

# Check schema.py has actual content
schema_content = (PROJECT / 'src/airbnb_serving/schema.py').read_text()
assert 'BaseModel' in schema_content, 'schema.py must define Pydantic models'

# Check app.py has endpoints
app_content = (PROJECT / 'src/airbnb_serving/app.py').read_text()
assert '/health' in app_content, 'app.py must have /health endpoint'
assert '/predict' in app_content, 'app.py must have /predict endpoint'
assert 'batch' in app_content, 'app.py must have /predict/batch endpoint'

print('All required files present.')
print('No credential leaks detected.')
print('HW03 proof passed.')

All required files present.
No credential leaks detected.
HW03 proof passed.


## Screenshots required

Add these to the `screenshots/` folder before submitting:

- `screenshots/health_endpoint.png` — GET /health response
- `screenshots/predict_endpoint.png` — POST /predict response
- `screenshots/batch_endpoint.png` — POST /predict/batch response
- `screenshots/fastapi_docs.png` — auto-generated docs at /docs
- `screenshots/docker_image_sizes.png` — output of `docker images` showing both image sizes

## Commit

```bash
git add .
git commit -m "HW03 model serving and deployment"
git push
```